<a href="https://colab.research.google.com/github/smduarte/spbd-2627/blob/main/docs/labs/lab2/SPBD_Labs_python1_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1 - Environment & Python Exercises (Solutions)

##1. Word Frequency

Create a notebook program that counts the frequency of each word in a text file (for example, the novel "Os Maias").

a) No sorting required: words and their frequencies can appear in any order.

b) Sorted by word, in ascending alphabetical order.

c) Sorted by frequency, in descending order (most frequent words first).

In [ ]:
#@title Download Dataset
!wget -q -O os_maias.txt https://www.dropbox.com/s/n24v0z7y79np319/os_maias.txt?dl=0

### a) No sorting required

In [ ]:
import re

# Read the whole text. Lowercase so "O" and "o" count as the same word.
with open("os_maias.txt", encoding="utf-8") as f:
    text = f.read().lower()

# [^\W_]+ = runs of letters/digits (Unicode-aware, so á, ç, ã... are kept).
# Punctuation and underscores are dropped and act as word separators.
words = re.findall(r"[^\W_]+", text)

freq = {}
for w in words:
    freq[w] = freq.get(w, 0) + 1

for word, count in freq.items():
    print(f"{word}\t{count}")

### b) Sorted by word (ascending alphabetical order)

In [ ]:
import re

with open("os_maias.txt", encoding="utf-8") as f:
    text = f.read().lower()

words = re.findall(r"[^\W_]+", text)

freq = {}
for w in words:
    freq[w] = freq.get(w, 0) + 1

for word, count in sorted(freq.items()):
    print(f"{word}\t{count}")

### c) Sorted by frequency (descending)


In [ ]:
import re

with open("os_maias.txt", encoding="utf-8") as f:
    text = f.read().lower()

words = re.findall(r"[^\W_]+", text)

freq = {}
for w in words:
    freq[w] = freq.get(w, 0) + 1

for word, count in sorted(freq.items(), key=lambda kv: -kv[1]):
    print(f"{word}\t{count}")

## 2. Weblog Analysis

Consider a set of log files captured during a DDoS (Distributed Denial of Service) attack, containing web access information recorded by the server during the attack.

Create a python notebook that processes these web log entries to:

a) Count the number of unique IP addresses involved in the DDoS attack.

b) For each 10-second interval, calculate the following metrics: [number of requests, average execution time, maximum execution time, minimum execution time].

c) Build an inverted index that maps each 10-second interval to a list of unique IP addresses that made requests (grouped by URL/Path).

The log files contain tab-separated text lines formatted as follows:


The log files contain text lines as shown below, with blank as the separator:

date |IP_source | status_code | operation | URL/Path | execution time |
-|-|-|-|-|-
timestamp  | string | int | string | string| float |
2016-12-06T08:58:35.318+0000|37.139.9.11|404|GET|/codemove/TTCENCUFMH3C|0.026

In [ ]:
#@title Download Dataset
!wget -q -O weblog.txt https://www.dropbox.com/s/0r8902uj9yum7dg/web.log?dl=0
!head -1 weblog.txt

### a) Number of unique IP addresses

In [ ]:
unique_ips = set()
with open("weblog.txt", encoding="utf-8", errors="ignore") as f:
  lines = f.readlines()
  for line in lines:
    parts = line.strip().split(' ')
    if len(parts) == 6:
      unique_ips.add( parts[1] )

print("Unique IPs:", len(unique_ips))

### b) Per 10-second interval: [requests, average, maximum, minimum execution time]

In [ ]:
#@title Generic: Parsing the timestamp to map to an interval

from datetime import datetime

def to_interval( timestamp, duration ):
  dt = datetime.fromisoformat(timestamp)
  epoch_seconds = (int)(dt.timestamp()/duration) * duration
  return datetime.fromtimestamp(epoch_seconds)

records = dict()
with open("weblog.txt", encoding="utf-8", errors="ignore") as f:
  lines = f.readlines()
  for line in lines:
    parts = line.strip().split(' ')
    if len(parts) == 6:
      timestamp = parts[0]
      interval = to_interval( timestamp, 10)
      exec_time = float(parts[5])
      records.setdefault(interval, [] ).append( exec_time )

for interval, values in records.items():
  reqs = len(values)
  min_t = min(values)
  avg_t = sum(values) / reqs
  max_t = max(values)
  print(f"{interval}, #requests: {reqs}, (min: {min_t}, avg: {avg_t}, max: {max_t})")

In [ ]:
#@title Specific for 10-second intervals: slice the timestamp string on a 10-second boundary
records = dict()
with open("weblog.txt", encoding="utf-8", errors="ignore") as f:
  lines = f.readlines()
  for line in lines:
    parts = line.strip().split(' ')
    if len(parts) == 6:
      exec_time = float(parts[5])
      interval = parts[0][0:18]
      records.setdefault(interval, [] ).append( exec_time )

for interval, values in records.items():
  reqs = len(values)
  min_t = min(values)
  avg_t = sum(values) / reqs
  max_t = max(values)
  print(f"{interval}, #requests: {reqs}, (min: {min_t}, avg: {avg_t}, max: {max_t})")

### c) Inverted index: interval → URL/Path → unique IP addresses

In [ ]:
inverted_index = dict()
with open("weblog.txt", encoding="utf-8", errors="ignore") as f:
  lines = f.readlines()
  for line in lines:
    parts = line.strip().split(' ')
    if len(parts) == 6:
      ip = parts[1]
      url = parts[4]
      interval = parts[0][0:18]
      inverted_index.setdefault(f"{interval}-{url}", set() ).add( ip )

for key, ips in inverted_index.items():
  print(f"{key}, ips: {list(ips)}")